# Modeling Template

This is a template made for loading data from thousands of random Pokemon generation 9 battles from https://pokemonshowdown.com/ and using features from a csv to create models which predict the winner. 

# Outline
## Section 1: Modeling
### 1.1 Logistic Regression
    1.1.1 Baseline model
    1.1.2 Improvements
### 1.2 Decision Tree
### 1.3 Random Forest
### 1.4 
### 1.5
## Section 2: Model Comparisons
### 2.1 AUC score
### 2.2 Cross Validation Score
## Section 3: Best Model Analysis
### 3.1 ROC Curve
### 3.2 Confusion Matrix

In [1]:
# Import statements
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import copy

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score,roc_curve,accuracy_score,confusion_matrix
from sklearn.model_selection import cross_val_score,train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator,ClassifierMixin
from xgboost import XGBClassifier

sns.set_style("whitegrid")

ImportError: scipy._cyutility does not export expected C function slice_memviewslice

In [ ]:
# Load in training dataframe
path_to_train = "../data/data_cleaned.csv.zip"
full_df = pd.read_csv(path_to_train)

df_copy = copy.deepcopy(full_df)

# get speed difference as a feature
p1_speed_cols = [f"M1{i}_spe" for i in range(1, 7)]
p2_speed_cols = [f"M2{i}_spe" for i in range(1, 7)]
full_df['speed_diff'] = df_copy[p1_speed_cols].sum(axis=1) - df_copy[p2_speed_cols].sum(axis=1)

# Eliminate any matches where one player quits prematurely
complete_matches = full_df[(full_df['duration'] > 60) & ((full_df["p1_revealed_team_size"] > 2) | (full_df["p2_revealed_team_size"] > 2))]

# only analyze matches for which we know the players' elo ratings (so that we can use elo_diff as a feature)
train_df = complete_matches[complete_matches['p1elo0'] > 0].copy()
train_df['elo_diff'] =  train_df['p1elo0'] - train_df['p2elo0']

print(train_df.columns.tolist())

# Load in testing dataframe
# path_to_test = "<replace with path to test data csv>"
# test_df = pd.read_csv(path_to_test)
# print(test_df.columns.tolist())

# Section 1: Modeling
## 1.1 Logistic Regression
### 1.1.1 Baseline Model

We begin with a simple model using only one feature to predict battle outcome. Following models will add features to see if this baseline model can be improved.

In [ ]:
class CustomEloPredictor(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"

    def __init__(self, scale=400):
        self.scale = scale

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]
        return self

    def predict_proba(self, X):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        elo_diff = X[:, 0].astype(float)
        p = 1 / (1 + 10 ** (-elo_diff / self.scale))
        return np.column_stack([1 - p, p])

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        preds = (probs >= 0.5).astype(int)
        return self.classes_[preds]

    def score(self, X, y):
        return np.mean(self.predict(X) == y)

    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.estimator_type = "classifier"
        return tags

In [ ]:
########## BASELINE MODEL ##########
base_feature = ["elo_diff"]
X_train_base = train_df[base_feature]
y_train = train_df["p1_win"]

custom_baseline = CustomEloPredictor()


### 1.1.2 Improvements

In [ ]:
########## IMPROVED MODEL ##########

# This is the full set of features that improve some models.  Not every model uses all of these features.  The best model uses all but total_stat_diff and type_diversity_diff.
features = [
    "elo_diff",
    "p1_total_adv",
    "total_stat_diff",
    "speed_diff",
    "type_diversity_diff",
    "num_move_boosters_diff",
    "num_boosting_abilities_diff"
]
X_train = train_df[features]

# preprocess = ColumnTransformer(
#     transformers=[
#         ("scaler", StandardScaler(), ["elo_diff"])
#     ],
#     remainder="passthrough"
# )
preprocess = StandardScaler()

lr_balanced = Pipeline([
    ('preprocess',preprocess),
    ('lr',LogisticRegression(max_iter=10000,C=np.inf,fit_intercept=False))
])

lr_biased = LogisticRegression(max_iter=10000,C=np.inf,fit_intercept=True)

## 1.2 Decision Tree

In [ ]:
# Do a grid search for decision tree parameters
dt_param_grid = {
    "max_depth": [3, 5, 8, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=0),
    param_grid=dt_param_grid,
    cv=5,
    scoring="neg_log_loss",
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

print("Best parameters:")
print(dt_grid.best_params_)

In [ ]:
dt = Pipeline([
    ('preprocess',preprocess),
    ('dt',DecisionTreeClassifier(
    random_state=0,
    max_depth = 3,
    min_samples_leaf= 1,
    min_samples_split= 20))
])

## 1.3 Random Forest

In [ ]:
# Do a grid search for random forest parameters
rf_param_grid = {
    "max_depth": [4, 6, 8, 10],
    "min_samples_leaf": [5, 10, 20],
    "n_estimators": [200, 500, 800]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=0),
    param_grid=rf_param_grid,
    cv=5,
    scoring="neg_log_loss",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

In [ ]:
rf = Pipeline([
    ('preprocess',preprocess),
    ('rf',RandomForestClassifier(
    max_depth=4,
    min_samples_leaf=20,
    n_estimators=500,
    random_state=0))
])

## 1.4 Histogram-based Gradient Boosting Classification Tree

In [ ]:
# Do a grid search for HistGradientBoosting parameters
hgb_param_grid = {
    "max_depth": [4, 6, 8, 10],
    "min_samples_leaf": [5, 10, 20],
    "max_iter": [100, 200, 500] 
}

grid = GridSearchCV(
    HistGradientBoostingClassifier(random_state=0),
    param_grid=hgb_param_grid,
    cv=5,
    scoring="neg_log_loss",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

In [ ]:
hgb = Pipeline([
    ('preprocess',preprocess),
    ('hgb',HistGradientBoostingClassifier(
    max_depth=4,
    max_iter=100,
    min_samples_leaf=5,
    random_state=0))
])

## 1.5 XGBoost

In [ ]:
# Do a grid search for XGBoost parameters
xgb_param_grid = {
    "max_depth": [4, 6, 8, 10],
    "min_child_weight": [5, 10, 20], 
    "n_estimators": [200, 500, 800] 
}

grid = GridSearchCV(
    XGBClassifier(random_state=0),
    param_grid=xgb_param_grid,
    cv=5,
    scoring="neg_log_loss",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

In [ ]:
xgb = Pipeline([
    ('preprocess',preprocess),
    ('xgb',XGBClassifier(
    max_depth=4,
    min_child_weight=20,
    n_estimators=200,
    random_state=0))
])

# Section 2: Model Comparisons

## 2.1 Cross Validation Score

In [ ]:
models = [
    ("custom baseline",custom_baseline, X_train_base),
    ("full unbiased linear regression", lr_balanced, X_train[['elo_diff','p1_total_adv','num_move_boosters_diff','num_boosting_abilities_diff']]),
    ("full biased linear regression", lr_biased, X_train)
    # ("decision tree", dt, X_train),
    # ("random forest", rf, X_train),
    # ("histogram Gradient Boosting", hgb, X_train),
    # ("XGBoost", xgb, X_train)
]

for i in range(len(models)):
    model_name = models[i][0]
    model = models[i][1]
    data_frame = models[i][2]
    # get cross-validation scores
    cvscore = cross_val_score(model,data_frame,y_train,cv=5,n_jobs=-1,scoring="accuracy")
    print(f"The average accuracy score for the {model_name} model is {cvscore.mean()} +/- {cvscore.std(ddof=1)}")

    # get confusion matrices for a basic train_test_split
    data_frame_tt,data_frame_val,y_tt,y_val = train_test_split(data_frame,y_train,test_size=0.2,shuffle=True,random_state=207)
    model.fit(data_frame_tt,y_tt)
    preds = model.predict(data_frame_val)
    cm = confusion_matrix(y_val,preds,normalize='all')
    print(f"The confusion matrix for the {model_name} model is")
    print(cm)
    print()

# Section 3: Best Model Analysis
## 3.1 ROC Curve

#### Choose best model and uncomment it below
 - Logistic regression model = lr
 - Decision tree model = dt
 - Random forest model = rf

In [ ]:
# Choose best model
best_model_index = 1
best_model = models[best_model_index][1]
data_frame = models[best_model_index][2]
best_model.fit(data_frame,y_train)

In [ ]:
# Random forest
prob = best_model.predict_proba(data_frame)[:,1]
fpr, tpr, _ = roc_curve(y_train, prob)

plt.figure(figsize=(6,6))

plt.plot(
    fpr,
    tpr,
    label=f"Best Model (AUC={roc_auc_score(y_train, prob):.3f})"
)

plt.plot([0,1],[0,1],"k--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()

## 3.2 Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    best_model,
    data_frame,
    y_train,
    normalize='all'
)

## 3.3 Feature Separation

Let's see if our best model is predicting as it should along the various features.

In [ ]:
train_df["predictions"] = best_model.predict(data_frame)

fig,axs = plt.subplots(ncols=4,sharey=False,figsize=(18,6))
sns.histplot(data = train_df,x='elo_diff',hue='predictions',ax=axs[0])
sns.histplot(data=train_df,x='p1_total_adv',hue='predictions',ax=axs[1])
sns.histplot(data = train_df, x = 'num_boosting_abilities_diff',hue='predictions',ax=axs[2])
sns.histplot(data=train_df,x='num_move_boosters_diff',hue='predictions',ax=axs[3])

for i in range(4):
    axs[i].set_ylabel("Number of Battles")
    axs[i].legend(labels=["P1 predicted win","P1 predicted loss"])

plt.tight_layout()
plt.show()
